# Modèles génératifs en astronomie : améliorer les images de télescope avec l’IA 🚀🔭
*Par: Gabriel Missael Barco, Nicolas Payot, Auriane Thilloy, Olivia Pereira*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/GabrielMissael/super-resolution-workshop/blob/master/notebooks/Diffusion_Simulated_Galaxy_Pipeline_fr.ipynb
)      [![View on GitHub](https://img.shields.io/badge/View_on-GitHub-black?logo=github)](
https://github.com/GabrielMissael/super-resolution-workshop
)

Bienvenue ! Dans ce notebook, nous découvrirons comment l’**astronomie** et l’**apprentissage automatique** peuvent travailler ensemble.
Les images obtenues de télescopes peuvent être floues, bruitées et de faible résolution. Nous apprendrons à :

1. Simuler la manière dont un télescope déforme l’image nette d’une galaxie.
2. Utiliser un important modèle d’IA (**modèle de diffusion**) ayant appris la forme habituelle de galaxies.
3. Combiner ces deux éléments pour reconstruire une image nette et haute résolution d’une galaxie à partir d’une observation bruitée.
4. Terminer avec un défi sur une **galaxie mystère**. 👀

Vous n’avez besoin que des bases en Python. Nous garderons les mathématiques légères et nous nous concentrerons sur les **idées** et leur **présentation**.


## 0. Code caché 🧰

Cette cellule définit tous les **outils** utilisés en coulisses pour la suite :

- Des fonctions pour afficher les images,
- Du code qui simule les effets du télescope (flou, sous-échantillonnage, bruit),
- Et une petite technique d’inférence qui communique avec notre modèle de diffusion.

Vous n’avez pas besoin de lire ou de comprendre toute cette cellule pour suivre le notebook.
Voyez-la plutôt comme le **moteur sous le capot**. Nous conduirons la voiture dans les prochaines cellules 🚗✨.

In [ ]:
!git clone --quiet https://github.com/GabrielMissael/super-resolution-workshop
!pip3 install -q git+https://github.com/AlexandreAdam/score_models.git@dev

In [ ]:
import sys
sys.path.append("super-resolution-workshop")

from src.diffusion_sampling.diffusion_sampling import *

## 1. Chargement de nos données de galaxies et du modèle d’IA 🌌🤖

Nous téléchargeons maintenant deux éléments importants :

- Un petit ensemble d’**images de galaxies** (ce à quoi le ciel ressemble vraiment dans notre univers simplifié).
- Un **modèle de diffusion** préentraîné qui a appris le « langage des galaxies » à partir de nombreux exemples.

À la fin de la cellule, nous visualiserons rapidement quelques galaxies pour nous familiariser avec les données que nous allons utiliser.

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download
from score_models import ScoreModel
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Using device:", DEVICE)

# Where we want the model to live in the notebook filesystem
MODEL_DIR = Path("model/galaxy_prior")

if not MODEL_DIR.exists():
    print("Downloading galaxy prior from Hugging Face…")
    snapshot_download(
        repo_id="GMissaelBarco/galaxy-prior",
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,  # safer on Colab
    )

# Load pre-trained diffusion model (galaxy prior)
model = ScoreModel(path=str(MODEL_DIR)).to(DEVICE)
model.load()
model.eval()
print("Model loaded ✓")


In [ ]:
# Load galaxy dataset
galaxies = torch.load("super-resolution-workshop/data/galaxies.pt", map_location=DEVICE)
print("Galaxies shape:", galaxies.shape)

# Quick look at the first 5 galaxies
show_grid(galaxies, title="Example clean galaxies")


## 2. Effet de télescope #1 : le flou (la PSF) 👓

Les télescopes réels ne produisent jamais des images parfaitement nettes. À cause de l’optique et de l’atmosphère, la lumière provenant d’un point s’étale. C’est ce qu’on décrit avec la fonction d’étalement du point (PSF).

Dans cette cellule interactive, vous pouvez :

- Modifier la largeur de la PSF (`Sigma PSF`),
- Voir comment les galaxies deviennent plus ou moins floues.

Amusez-vous avec le curseur et observez comment l’augmentation de la PSF étale les détails des bras spiraux.

In [ ]:
sigma_psf_slider_explore = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=5.0,
    step=0.01,
    description="Sigma PSF:",
    layout=widgets.Layout(width="800px"),
)

out_psf = widgets.Output()

def update_psf_explore(change=None):
    with out_psf:
        clear_output(wait=True)
        psf_images = psf_on_image(galaxies, sigma=float(sigma_psf_slider_explore.value))
        show_grid(psf_images, title="Effect of PSF (blur only)")

sigma_psf_slider_explore.observe(update_psf_explore, names="value")
display(sigma_psf_slider_explore)
update_psf_explore()
display(out_psf)


## 3. Effet de télescope #2 : résolution limitée 🔍➡️🧱

Les détecteurs sont constitués de **pixels**, et leur nombre est limité.
Si nous utilisons moins de pixels, nous perdons de fins détails et ceci, même sans appliquer de flou.

Ici, vous pouvez :

- Modifier le nombre de pixels conservés (`Pixels downsampled to`),
- Voir comment une même galaxie apparaît à différentes résolutions.

Remarquez comment les petits détails disparaissent à mesure que la résolution diminue.

In [ ]:
pixels_downsample_slider_explore = widgets.IntSlider(
    value=64,
    min=8,
    max=64,
    step=1,
    description="Pixels downsampled to:",
    layout=widgets.Layout(width="800px"),
)

out_downsample = widgets.Output()

def update_downsample_explore(change=None):
    with out_downsample:
        clear_output(wait=True)
        downsampled = downsample_img(galaxies, size=int(pixels_downsample_slider_explore.value))
        show_grid(downsampled, title="Effect of downsampling (resolution only)")

pixels_downsample_slider_explore.observe(update_downsample_explore, names="value")
display(pixels_downsample_slider_explore)
update_downsample_explore()
display(out_downsample)


## 4. Effet de télescope #3 : le bruit aléatoire 🌧️

Même avec un télescope parfait, nos images sont affectées par du **bruit** :

- Les photons arrivent de manière aléatoire,
- L’électronique du détecteur ajoute aussi sa propre part d’aléatoire.

Cette cellule vous permet de :

- Augmenter ou diminuer le **niveau de bruit**,
- Voir comment l’image devient granuleuse et plus difficile à interpréter.

Essayez un bruit élevé et imaginez à quel point il serait difficile de mesurer la forme des galaxies à partir de telles données !

In [ ]:
sigma_noise_slider_explore = widgets.FloatSlider(
    value=0.00,
    min=0.0,
    max=0.5,
    step=0.0005,
    description="Noise σ:",
    layout=widgets.Layout(width="800px"),
)

out_noise = widgets.Output()

def update_noise_explore(change=None):
    with out_noise:
        clear_output(wait=True)
        noisy = add_gaussian_noise(galaxies, sigma=float(sigma_noise_slider_explore.value))
        show_grid(noisy, title="Effect of Gaussian noise")

sigma_noise_slider_explore.observe(update_noise_explore, names="value")
display(sigma_noise_slider_explore)
update_noise_explore()
display(out_noise)


## 5. Combiner les effets du télescope 🧪

En réalité, tous ces effets se produisent en **même temps** :

1. La galaxie est floutée par la PSF,
2. Sa lumière est enregistrée sur une grille de pixels finie (sous-échantillonnage),
3. Du bruit vient s’ajouter par-dessus.

Les curseurs ci-dessous contrôlent ces trois étapes. Le panneau affiche :

- Cinq galaxies après **flou + sous-échantillonnage + bruit**,
- Un cadre vert qui met en évidence la galaxie que nous essaierons de reconstruire plus tard.

Amusez-vous avec les curseurs pour créer une configuration de télescope réaliste (ou extrême !), puis choisissez votre galaxie préférée pour la tâche de reconstruction.

Selon vous, quel paramètre (PSF, résolution, bruit) détériore le plus l’image ? Pourquoi ?

In [ ]:
# Shared sliders for the forward model used later in inference
sigma_psf_slider = widgets.FloatSlider(
    value=0.01,
    min=0.01,
    max=5.0,
    step=0.01,
    description="Sigma PSF:",
    layout=widgets.Layout(width="800px"),
)

pixels_downsample_slider = widgets.IntSlider(
    value=64,
    min=10,
    max=64,
    step=1,
    description="Pixels downsampled to:",
    layout=widgets.Layout(width="800px"),
)

sigma_noise_slider = widgets.FloatSlider(
    value=0.01,
    min=0.01,
    max=0.2,
    step=0.0005,
    description="Noise σ:",
    layout=widgets.Layout(width="800px"),
)

image_selector = widgets.ToggleButtons(
    options=[("Image 1", 0), ("Image 2", 1), ("Image 3", 2), ("Image 4", 3), ("Image 5", 4)],
    description="Select image:",
    layout=widgets.Layout(width="800px"),
)

out_pipeline = widgets.Output()

# Global variables to reuse in later cells
galaxies_psf = None
galaxies_downsampled = None
galaxies_noisy = None

def update_pipeline(change=None):
    global galaxies_psf, galaxies_downsampled, galaxies_noisy

    with out_pipeline:
        clear_output(wait=True)

        sigma_psf_val = float(sigma_psf_slider.value)
        size_val = int(pixels_downsample_slider.value)
        sigma_n_val = float(sigma_noise_slider.value)

        galaxies_psf = psf_on_image(galaxies, sigma=sigma_psf_val)
        galaxies_downsampled = downsample_img(galaxies_psf, size=size_val)
        galaxies_noisy = add_gaussian_noise(galaxies_downsampled, sigma_n_val)

        selected_idx = image_selector.value
        show_grid_final(galaxies_noisy, selected_idx=selected_idx)

for w in [sigma_psf_slider, pixels_downsample_slider, sigma_noise_slider, image_selector]:
    w.observe(update_pipeline, names="value")

ui = widgets.VBox(
    [
        sigma_psf_slider,
        pixels_downsample_slider,
        sigma_noise_slider,
        image_selector,
        out_pipeline,
    ]
)

display(ui)
update_pipeline()


## 6. Une minute sur les modèles de diffusion 🌫️➡️🌌

Notre modèle d’IA est un **modèle de diffusion**, un type de modèle génératif capable de créer de nouvelles galaxies.

L’idée, en quelques mots :

1. On part d’une image réelle de galaxie et on **ajoute progressivement du bruit** jusqu’à obtenir un signal qui ressemble à du pur bruit.
2. On entraîne un réseau de neurones à inverser ce processus, en retirant le bruit étape par étape pour retrouver les structures.
3. Une fois le modèle entraîné, on peut partir d’un bruit totalement aléatoire et laisser le modèle débruiter l'image petit à petit jusqu’à produire une galaxie réaliste.

Dans la prochaine cellule, nous allons échantillonner directement depuis ce modèle prior pour voir quels types de galaxies il a appris à générer 🎨.


In [ ]:
# Prior sampling code
prior_samples = model.sample(shape=(20, 1, 64, 64), device=DEVICE, steps=70)

fig, ax = plt.subplots(2, 10, figsize=(20, 4), dpi=200)
for i in range(2):
    for j in range(10):
        idx = i * 10 + j
        img = img_to_show(prior_samples[idx], log_scale=True)
        ax[i, j].imshow(img, cmap="magma")
        ax[i, j].axis("off")
plt.suptitle("Samples from the diffusion model prior", fontsize=16)
plt.tight_layout()
plt.show()

## 7. De l’image floue à la galaxie nette : le problème inverse 🔄

Voici maintenant le défi principal :

> À partir d’une image de galaxie **bruitée, floue et de faible résolution**, peut-on deviner à quoi ressemblait la galaxie **haute résolution** sous-jacente ?

Pour cela, nous :

1. Construisons un opérateur linéaire `A` qui applique les mêmes effets de télescope (PSF + sous-échantillonnage) utilisés plus haut.
2. Utilisons notre modèle de diffusion comme _**prior**_ (il "connaît" à quoi ressemblent les galaxies en général).
3. Exécutons un algorithme d’échantillonnage (`LinearGaussianPosteriorSampler`) qui combine les données et le _prior_ pour générer des galaxies haute résolution compatibles avec l’observation.

Cette cellule met en place `A`, sélectionne la galaxie que vous avez choisie plus tôt et lance l’échantillonneur pour produire plusieurs échantillons du posterior.

Si on essayait d'"annuler" les effets du télescope sans aucune connaissance préalable sur les galaxies, que se passerait-il à votre avis ?

In [ ]:
# Build the linear operator A corresponding to the current PSF + downsampling
sigma_psf_val = float(sigma_psf_slider.value)
S_val = int(pixels_downsample_slider.value)
y_lin, A = psf_downsample_build_A(galaxies, sigma_psf=sigma_psf_val, S=S_val)


# Pick the selected galaxy and its noisy observation
idx = image_selector.value
y_obs = galaxies_noisy[idx]    # (S,S)
sigma_n = float(sigma_noise_slider.value)

sampler = LinearGaussianPosteriorSampler(
    observation=y_obs,   # (S,S)
    A=A,
    model=model,
    sigma_n=sigma_n,
    C=1.0,
    M=0.0,
)

samples = sampler.run(
    n_samples=4,
    steps=100,
    progress=True,
    true=galaxies[idx],       # for optional trajectory plotting
    plot_trajectory=True,    # set True if you want the animated view
    trajectory_stride=5,
)

print("Samples shape:", samples.shape)  # (4,1,Hs,Hs)


## 8. Examiner la reconstruction 🎨

Il est temps d’inspecter ce que l’échantillonneur a produit :

- **Ligne du haut** : la véritable galaxie haute résolution (à gauche) et quelques **échantillons du _posterior_** représentant ce que la galaxie pourrait être.
- **Ligne du milieu** : l’**image observée réelle** (à gauche) et les **observations simulées** obtenues en faisant passer chaque échantillon à travers notre modèle de télescope `A`.
- **Ligne du bas** : les **résidus** = (observation − observation simulée) / niveau de bruit.

Si la méthode fonctionne bien, les observations simulées devraient ressembler à l’observation réelle, et les résidus devraient ressembler à du bruit aléatoire (i.e. sans motifs évidents).

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9), dpi=200)

# Row 1: true + posterior samples
true_img = img_to_show(galaxies[idx], log_scale=True)
axes[0, 0].imshow(true_img, cmap="magma")
axes[0, 0].set_title("True")
axes[0, 0].axis("off")

for i in range(4):
    img = img_to_show(samples[i], log_scale=True)
    axes[0, i + 1].imshow(img, cmap="magma")
    axes[0, i + 1].set_title(f"Sample {i+1}")
    axes[0, i + 1].axis("off")

# Row 2: observed + mock observations
obs_img = img_to_show(y_obs, log_scale=False)
axes[1, 0].imshow(obs_img, cmap="magma")
axes[1, 0].set_title("Observed")
axes[1, 0].axis("off")

for i in range(4):
    mock = samples[i].view(1, sampler.Msrc) @ A.t()  # (1,Mobs)
    mock_img = mock.view(1, 1, y_obs.shape[0], y_obs.shape[1])
    mock_img_to_show = img_to_show(mock_img[0, 0], log_scale=False)
    axes[1, i + 1].imshow(mock_img_to_show, cmap="magma")
    axes[1, i + 1].set_title(f"Mock Obs {i+1}")
    axes[1, i + 1].axis("off")

    # Row 3: residuals
    residual = (y_obs - mock_img[0, 0]) / sigma_n
    residual_img = img_to_show(residual, log_scale=False)
    axes[2, i + 1].imshow(residual_img, cmap="bwr", vmin=-3, vmax=3)
    axes[2, i + 1].set_title(f"Residual {i+1}")
    axes[2, i + 1].axis("off")

axes[2, 0].axis("off")
plt.tight_layout()
plt.show()


## 9. Image moyenne et carte d’incertitude 📊

Au lieu d’examiner les échantillons un par un, nous pouvons résumer le _posterior_ avec :

- La **moyenne du _posterior_** (la moyenne de tous les échantillons) : une image unique représentant notre "meilleure estimation" de la galaxie.
- L’**écart-type du _posterior_**: une **carte d’incertitude** indiquant les zones où le modèle est plus ou moins sûr.

Comparez :

- La galaxie réelle,
- La moyenne du _posterior_,
- La carte d’incertitude.

In [ ]:
samples_mean = samples.mean(dim=0)  # (1,Hs,Hs)
samples_std = samples.std(dim=0)    # (1,Hs,Hs)

fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=200)

img_mean = img_to_show(samples_mean[0], log_scale=True)
axes[1].imshow(img_mean, cmap="magma")
axes[1].set_title("Posterior Mean")
axes[1].axis("off")

img_std = img_to_show(samples_std[0], log_scale=False)
axes[2].imshow(img_std, cmap="magma")
axes[2].set_title("Posterior Std Dev")
axes[2].axis("off")

true_img = img_to_show(galaxies[idx], log_scale=True)
axes[0].imshow(true_img, cmap="magma")
axes[0].set_title("True Image")
axes[0].axis("off")

plt.tight_layout()
plt.show()


## 10. Le défi de la galaxie mystère 🕵️‍♀️🌌

Jusqu’ici, nous avons travaillé avec des galaxies dont nous connaissions secrètement les images réelles.
Passons maintenant à quelque chose de plus intéressant :

> Des astronomes ont observé une **galaxie mystère** à 12 reprises avec un petit télescope.
> Chaque observation est floue et bruitée, et nous ne regarderons **pas** l’image haute résolution réelle avant la toute fin.

Votre mission :
Utiliser le même pipeline IA + physique pour reconstruire l’apparence la plus probable de cette galaxie. Prêt·e ? 🙂

### 10.1 Les données : 12 poses bruitées 📷📷📷

Dans cette cellule, nous chargeons les 12 images de télescope de notre galaxie mystère.

Chaque panneau représente :

- Une **photographie** indépendante du même objet,
- Avec le **même télescope** et le **même niveau de bruit**.

Pour l’instant, vous ne voyez qu’une tache bruitée… mais une forme très distinctive s’y cache.
Regardez la grille et essayez d’imaginer quel type de galaxie pourrait se dissimuler derrière tout ce bruit.

In [ ]:
#! Do not change these values!!
sigma_n_ood = 0.1
sigma_psf_ood = 0.1
res_ood = 64

# Load observations of the mystery OOD galaxy
mistery_galaxy_obs = torch.load("super-resolution-workshop/data/mistery_galaxy_obs.pt")

fig, ax = plt.subplots(3, 4, figsize=(16, 12), dpi=200)
for i in range(12):
    img = img_to_show(mistery_galaxy_obs[i], log_scale=False)
    ax[i // 4, i % 4].imshow(img, cmap="magma")
    ax[i // 4, i % 4].set_title(f"Mistery galaxy - Observation {i+1}")
    ax[i // 4, i % 4].axis("off")
plt.tight_layout()
plt.show()


### 10.2 Construire le modèle de télescope pour la galaxie mystère 🔧

Pour analyser ce nouveau sujet, nous avons besoin du **modèle de simulation** correspondant :

- Nous construisons un nouvel opérateur `A` qui reproduit la PSF et la résolution utilisées pour ces observations.
- Appliquer `A` à une galaxie nette nous donne une version simulée, sans bruit, de ce que le télescope verrait.

C’est la même idée qu’auparavant, mais, cette fois, l’opérateur est calibré spécifiquement pour les données de la galaxie mystère.

In [ ]:
sigma_n_ood = 0.1
sigma_psf_ood = 0.1
res_ood = 64

y_lin_ood, A = psf_downsample_build_A(
    mistery_galaxy_obs,
    sigma_psf=sigma_psf_ood,
    S=res_ood,
)

print("A shape:", A.shape)
print("y_lin shape:", y_lin_ood.shape)


### 10.3 Échantillonner des galaxies possibles 🎲

Nous relions maintenant tous les éléments :

1. Les **12 observations** (une par photographie) sont combinées dans le _likelihood_.
2. Le modèle de diffusion fournit un _**prior**_ sur les galaxies réalistes.
3. L’échantillonneur génère plusieurs **échantillons du _posterior_** : différentes galaxies haute résolution qui sont toutes compatibles avec les données.

Chaque échantillon représente une reconstruction plausible de la galaxie mystère.
Lançons l’échantillonneur et voyons ce qu’il propose.

In [ ]:
sampler_mistery = LinearGaussianPosteriorSampler(
    observation=mistery_galaxy_obs,   # (B,res_ood,res_ood)
    A=A,
    model=model,
    sigma_n=sigma_n_ood,
    C=1.0,
    M=0.0,
)

samples_mistery = sampler_mistery.run(
    n_samples=4,
    steps=200,
    progress=True,
    true=None,
    plot_trajectory=True,
    trajectory_stride=5,
)

print("OOD samples shape:", samples_mistery.shape)  # (4,1,Hs,Hs)


### 10.4 Nos reconstructions expliquent-elles les données ? 👀

Cette figure reprend la même logique que précédemment :

- **Ligne du haut** : quatre échantillons du _posterior_ représentant la *véritable galaxie inconnue* (nous cachons toujours l’image réelle !).
- **Ligne du milieu** : l’une des images observées et les **observations simulées** générées à partir de chaque échantillon.
- **Ligne du bas** : les résidus pour cet échantillon.

Vérifiez si :

- Les observations simulées ressemblent bien à l’observation réelle,
- Les résidus ressemblent principalement à du bruit aléatoire.

Si c’est le cas, notre modèle combinant IA + physique fournit une très bonne explication des mesures.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9), dpi=200)

# Top row: OOD posterior samples
axes[0, 0].axis("off")
for i in range(4):
    img = img_to_show(samples_mistery[i], log_scale=True, min_val=0.1)
    axes[0, i + 1].imshow(img, cmap="magma")
    axes[0, i + 1].set_title(f"Sample {i+1}")
    axes[0, i + 1].axis("off")

# Middle row: Mistery galaxy observations and mock observations
obs_img = img_to_show(mistery_galaxy_obs[0], log_scale=False)
axes[1, 0].imshow(obs_img, cmap="magma")
axes[1, 0].set_title("Observed")
axes[1, 0].axis("off")

for i in range(4):
    mock = samples_mistery[i].view(1, sampler_mistery.Msrc) @ A.t()  # (1,Mobs)
    mock_img = mock.view(1, 1, mistery_galaxy_obs.shape[1], mistery_galaxy_obs.shape[2])
    mock_img_to_show = img_to_show(mock_img[0, 0], log_scale=False)
    axes[1, i + 1].imshow(mock_img_to_show, cmap="magma")
    axes[1, i + 1].set_title(f"Mock Obs {i+1}")
    axes[1, i + 1].axis("off")

    residual = (mistery_galaxy_obs[0] - mock_img[0, 0]) / sigma_n_ood
    residual_img = img_to_show(residual, log_scale=False)
    axes[2, i + 1].imshow(residual_img, cmap="bwr", vmin=-3, vmax=3)
    axes[2, i + 1].set_title(f"Residual {i+1}")
    axes[2, i + 1].axis("off")

# Question mark for unknown true image
axes[0, 0].text(0.5, 0.5, "?", fontsize=40, ha="center", va="center")
axes[0, 0].set_title("True (unknown)")

axes[2, 0].axis("off")
plt.tight_layout()
plt.show()


### 10.5 Le grand dévoilement 🎉🍁

Il est temps de lever le voile !

Dans cette cellule, nous chargeons enfin la **véritable image haute résolution** de la galaxie mystère et la comparons à nos échantillons du posterior.

Surprise ! La "galaxie" a été conçue pour ressembler à une **feuille d’érable**. Un petit clin d’œil canadien. 🍁

Observez attentivement :

- Les amas lumineux et la **forme générale de la feuille** dans les échantillons correspondent-ils à l’image réelle ?
- Certaines parties de la feuille (tige, pointes, bords) sont-elles mieux capturées par certains échantillons que par d’autres ?

Cela montre comment notre modèle combinant IA + physique peut retrouver une forme très spécifique à partir de données bruitées et floutées, et ce, même sans avoir jamais vu de feuille d'érable.

In [ ]:
true_mistery = torch.load(
    "super-resolution-workshop/data/true_mistery_galaxy.pt"
).to(DEVICE)

# Plot posterior samples against the true mistery galaxy
fig, axes = plt.subplots(1, 5, figsize=(15, 3), dpi=200)
true_img = img_to_show(true_mistery[0], log_scale=True, min_val=0.1)
axes[0].imshow(true_img, cmap="magma")
axes[0].set_title("True Mistery Galaxy")
axes[0].axis("off")

for i in range(4):
    img = img_to_show(samples_mistery[i], log_scale=True, min_val=0.1)
    axes[i + 1].imshow(img, cmap="magma")
    axes[i + 1].set_title(f"Sample {i+1}")
    axes[i + 1].axis("off")
plt.tight_layout()
plt.show()

### 10.6 Reconstruction moyenne et comparaison finale ✅

Nous comparons ici trois images côte à côte :

1. La **véritable galaxie en forme de feuille d’érable**,
2. La **moyenne du _posterior_** (la moyenne de tous les échantillons),
3. L’une des **poses observées et bruitées**.

Remarquez que :

- L’observation est tellement bruitée que la forme de la feuille d’érable est presque invisible,
- La moyenne du _posterior_ est beaucoup plus nette et révèle clairement le **contour de la feuille et sa tige**,
- Certains petits détails diffèrent encore, nous rappelant que nous avons toujours une part **d’incertitude**.

D’une tache granuleuse à une feuille d’érable reconnaissable grâce aux données et un _prior_ appris.
Vous venez d’utiliser un modèle génératif de pointe pour améliorer des images de télescope et résoudre un véritable problème inverse ! 🛰️🍁


In [ ]:
samples_mistery_mean = samples_mistery.mean(dim=0)  # (1,Hs,Hs)

fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=200)

true_img = img_to_show(true_mistery, log_scale=True, min_val=0.2)
axes[0].imshow(true_img, cmap="magma")
axes[0].set_title("True Image")
axes[0].axis("off")

img_mean = img_to_show(samples_mistery_mean, log_scale=True, min_val=0.2)
axes[1].imshow(img_mean, cmap="magma")
axes[1].set_title("Posterior Mean")
axes[1].axis("off")

# Stacking result
axes[2].imshow(img_to_show(mistery_galaxy_obs[0], log_scale=False), cmap="magma")
axes[2].set_title("Observed (1st exposure)")
axes[2].axis("off")
plt.tight_layout()
plt.show()
